# 데이터 로드 & 확인

In [1]:
import pandas as pd

df = pd.read_csv('/home/ubuntu/projects/data/bbc-text.csv')
print(df.shape)
print(df['category'].value_counts())
df.head()

(2225, 2)
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


# 전처리 - 레이블 인코딩 + 토크나이징 + 패딩

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 레이블 인코딩
le = LabelEncoder()
y = le.fit_transform(df['category'])  # sport/tech/... → 0,1,2,3,4

# 텍스트 토크나이징
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])
X = tokenizer.texts_to_sequences(df['text'])
X = pad_sequences(X, maxlen=200, padding='post', truncating='post')

# 훈련/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)

I0000 00:00:1775021551.954939   68438 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775021552.074366   68438 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775021553.766258   68438 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


(1780, 200) (1780,)


# 모델

## RNN

In [29]:
model_rnn = Sequential([
    layers.Embedding(input_dim=10000, output_dim=32, input_length=200),
    layers.SimpleRNN(32),
    layers.Dense(5, activation='softmax')
])
model_rnn.summary()
model_rnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_rnn = model_rnn.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
model_rnn.evaluate(X_test, y_test)

Model: "sequential_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_26 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_16 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

...

I0000 00:00:1775023293.138378   68924 dot_merger.cc:481] Merging Dots in computation: sequential_26_1_simple_rnn_16_1_while_body_92072_grad_92227_const_0__.21.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775023293.138481   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_92711__.24


22/23 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2248 - loss: 1.6071

I0000 00:00:1775023294.954727   68922 dot_merger.cc:481] Merging Dots in computation: sequential_26_1_simple_rnn_16_1_while_body_92072_grad_92227_const_0__.21.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775023294.954820   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_92711__.24


23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.2430 - loss: 1.5977 - val_accuracy: 0.2275 - val_loss: 1.6015
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.5920 - loss: 1.3737 - val_accuracy: 0.3146 - val_loss: 1.5502
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.8399 - loss: 1.0898 - val_accuracy: 0.3511 - val_loss: 1.5227
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9473 - loss: 0.7337 - val_accuracy: 0.3624 - val_loss: 1.4488
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9923 - loss: 0.4317 - val_accuracy: 0.3876 - val_loss: 1.4140
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.9958 - loss: 0.2583 - val_accuracy: 0.3455 - val_loss: 1.4932
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.9979 - loss: 0.1598 - val_accuracy: 0.3567 - val_loss: 1.5204
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 1.0000 - loss: 0.1053 - val_accuracy: 0.3427 - val_loss: 1

[1.7007027864456177, 0.307865172624588]

## DNN

In [12]:
model_dnn = Sequential([
    layers.Embedding(input_dim=10000, output_dim=32, input_length=200),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(5, activation='softmax')
])
model_dnn.summary()
model_dnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_dnn = model_dnn.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
model_dnn.evaluate(X_test, y_test)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

...

I0000 00:00:1775021872.499426   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_15145__.14


16/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2292 - loss: 1.6040

I0000 00:00:1775021873.185272   68920 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_15145__.14


23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.3097 - loss: 1.5827 - val_accuracy: 0.3652 - val_loss: 1.5309
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8968 - loss: 1.1709 - val_accuracy: 0.7219 - val_loss: 1.1602
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9867 - loss: 0.3767 - val_accuracy: 0.8624 - val_loss: 0.5607
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9993 - loss: 0.0502 - val_accuracy: 0.9185 - val_loss: 0.3687
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0145 - val_accuracy: 0.9017 - val_loss: 0.3356
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0076 - val_accuracy: 0.9213 - val_loss: 0.3079
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 0.9270 - val_loss: 0.2928
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0040 - val_accuracy: 0.9242 - val_loss: 0.2869
Ep

[0.3608073592185974, 0.8719100952148438]

## LSTM

In [13]:
model_lstm = Sequential([
    layers.Embedding(input_dim=10000, output_dim=32, input_length=200),
    layers.LSTM(32),
    layers.Dense(5, activation='softmax')
])
model_lstm.summary()
model_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_lstm = model_lstm.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
model_lstm.evaluate(X_test, y_test)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2690 - loss: 1.5991 - val_accuracy: 0.2669 - val_loss: 1.5777
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3862 - loss: 1.5113 - val_accuracy: 0.3511 - val_loss: 1.3967
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.4017 - loss: 1.3225 - val_accuracy: 0.3904 - val_loss: 1.2981
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4263 - loss: 1.1971 - val_accuracy: 0.4298 - val_loss: 1.1963
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4775 - loss: 1.1019 - val_accuracy: 0.4916 - val_loss: 1.1336
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6278 - loss: 0.9977 - val_accuracy: 0.5506 - val_loss: 0.9854
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6412 - loss: 0.8996 - val_accuracy: 0.6011 - val_loss: 0.9823
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6980 - loss: 0.7725 - val_accuracy: 0.6573 - v

[0.9566910862922668, 0.6067415475845337]

In [16]:
import pandas as pd

names    = ['SimpleRNN', 'DNN', 'LSTM']
models   = [model_rnn,    model_dnn,    model_lstm]
histories = [history_rnn, history_dnn, history_lstm]

results = []
for name, model, hist in zip(names, models, histories):
    tr  = round(hist.history['accuracy'][-1]*100, 1)
    val = round(hist.history['val_accuracy'][-1]*100, 1)
    _, te = model.evaluate(X_test, y_test, verbose=0)
    results.append([name, tr, val, round(te*100, 1)])

df = pd.DataFrame(results, columns=['모델', 'Train Acc(%)', 'Val Acc(%)', 'Test Acc(%)'])
df

,모델,Train Acc(%),Val Acc(%),Test Acc(%)
0,SimpleRNN,100.0,30.1,28.5
1,DNN,100.0,92.7,87.2
2,LSTM,77.3,65.4,60.7


# RNN/DNN 과적합

In [ ]:
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd

def run_exp(model, X_tr, y_tr, X_te, y_te, epochs=10, callbacks=None):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_tr, y_tr, epochs=epochs, batch_size=64, 
                     validation_split=0.2, callbacks=callbacks, verbose=0)
    _, te = model.evaluate(X_te, y_te, verbose=0)
    tr  = round(hist.history['accuracy'][-1]*100, 1)
    val = round(hist.history['val_accuracy'][-1]*100, 1)
    return tr, val, round(te*100, 1)

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
results = []

## RNN 실험

In [20]:
# 기본
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.SimpleRNN(32), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test)
results.append(['RNN', '기본', tr, val, te])

# ① Dropout
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.SimpleRNN(32), layers.Dropout(0.3), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test)
results.append(['RNN', '① Dropout(0.3)', tr, val, te])

# ② epochs=5
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.SimpleRNN(32), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, epochs=5)
results.append(['RNN', '② epochs=5', tr, val, te])

# ③ EarlyStopping
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.SimpleRNN(32), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, callbacks=[es])
results.append(['RNN', '③ EarlyStopping', tr, val, te])

# ④ Dropout + EarlyStopping
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.SimpleRNN(32), layers.Dropout(0.3), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, callbacks=[es])
results.append(['RNN', '④ Dropout+ES', tr, val, te])

print("RNN 완료")

I0000 00:00:1775022336.572645   68924 dot_merger.cc:481] Merging Dots in computation: sequential_12_1_simple_rnn_7_1_while_body_44434_grad_44589_const_0__.21.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775022336.572741   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_45073__.24
I0000 00:00:1775022338.391266   68922 dot_merger.cc:481] Merging Dots in computation: sequential_12_1_simple_rnn_7_1_while_body_44434_grad_44589_const_0__.21.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775022338.391370   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_45073__.24
I0000 00:00:1775022350.238465   68926 dot_merger.cc:481] Merging Dots in computation: sequential_13_1_simple_rnn_8_1_while_body_48464_grad_48660_const_0__.22.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775022350.238556   68926 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_49144__.25
I0000 00:00:1775022352

RNN 완료


## DNN 실험

In [21]:
# 기본
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.Flatten(), layers.Dense(64,activation='relu'), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test)
results.append(['DNN', '기본', tr, val, te])

# ① Dropout
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.Flatten(), layers.Dense(64,activation='relu'), layers.Dropout(0.3), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test)
results.append(['DNN', '① Dropout(0.3)', tr, val, te])

# ② epochs=5
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.Flatten(), layers.Dense(64,activation='relu'), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, epochs=5)
results.append(['DNN', '② epochs=5', tr, val, te])

# ③ EarlyStopping
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.Flatten(), layers.Dense(64,activation='relu'), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, callbacks=[es])
results.append(['DNN', '③ EarlyStopping', tr, val, te])

# ④ Dropout + EarlyStopping
m = Sequential([layers.Embedding(10000,32,input_length=200), layers.Flatten(), layers.Dense(64,activation='relu'), layers.Dropout(0.3), layers.Dense(5,activation='softmax')])
tr,val,te = run_exp(m, X_train, y_train, X_test, y_test, callbacks=[es])
results.append(['DNN', '④ Dropout+ES', tr, val, te])

print("DNN 완료")

I0000 00:00:1775022426.822970   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_61740__.14
I0000 00:00:1775022427.498692   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_61740__.14
I0000 00:00:1775022431.199191   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_64878__.15
I0000 00:00:1775022432.295233   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_64878__.15
I0000 00:00:1775022436.827254   68921 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_67932__.14
I0000 00:00:1775022437.457199   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_67932__.14
I0000 00:00:1775022440.586551   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_70284__.14
I0000 00:00:1775022441.232882   68922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_702

DNN 완료


In [23]:
df = pd.DataFrame(results, columns=['모델', '구분', 'Train Acc(%)', 'Val Acc(%)', 'Test Acc(%)'])
base = {'RNN': results[0][4], 'DNN': results[5][4]}  # 각 모델 기본 Test Acc
df['기본 대비 차이'] = df.apply(lambda r: f"{r['Test Acc(%)'] - base[r['모델']]:+.1f}", axis=1)
df

,모델,구분,Train Acc(%),Val Acc(%),Test Acc(%),기본 대비 차이
0,RNN,기본,100.0,33.7,31.0,+0.0
1,RNN,① Dropout(0.3),99.9,42.4,45.6,+14.6
2,RNN,② epochs=5,99.5,32.9,35.1,+4.1
3,RNN,③ EarlyStopping,99.9,40.7,38.9,+7.9
4,RNN,기본,99.9,44.9,44.5,+13.5
5,RNN,① Dropout(0.3),100.0,44.9,42.9,+11.9
6,RNN,② epochs=5,99.9,28.1,26.5,-4.5
7,RNN,③ EarlyStopping,84.6,33.4,27.2,-3.8
8,RNN,④ Dropout+ES,71.9,30.6,22.5,-8.5
9,DNN,기본,100.0,91.3,88.8,+45.9


# 과적합 RNN

In [39]:
model_rnn2 = Sequential([
    layers.Embedding(input_dim=10000, output_dim=16, input_length=200),
    layers.SpatialDropout1D(0.2),  
    layers.SimpleRNN(32, dropout=0.2, recurrent_dropout=0.2),  # RNN 내부 dropout 추가
    layers.Dense(5, activation='softmax')  # Dropout 제거
])
model_rnn2.summary()
model_rnn2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_rnn2 = model_rnn2.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
model_rnn2.evaluate(X_test, y_test)

/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_31 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_7             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_21 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

I0000 00:00:1775023701.996722   68926 dot_merger.cc:481] Merging Dots in computation: sequential_31_1_simple_rnn_21_1_while_body_114818_grad_114983_const_0__.21.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775023701.996849   68926 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_115558__.26


21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.2205 - loss: 1.6928

I0000 00:00:1775023706.795122   68920 dot_merger.cc:481] Merging Dots in computation: sequential_31_1_simple_rnn_21_1_while_body_114818_grad_114983_const_0__.21.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1775023706.795242   68920 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_115558__.26


23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 150ms/step - accuracy: 0.2142 - loss: 1.6937 - val_accuracy: 0.2303 - val_loss: 1.6117
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.2051 - loss: 1.6864 - val_accuracy: 0.2528 - val_loss: 1.6178
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.2065 - loss: 1.6912 - val_accuracy: 0.2500 - val_loss: 1.6600
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2212 - loss: 1.6818 - val_accuracy: 0.2416 - val_loss: 1.6429
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.2381 - loss: 1.6698 - val_accuracy: 0.2528 - val_loss: 1.6297
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2275 - loss: 1.6456 - val_accuracy: 0.2472 - val_loss: 1.6086
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2374 - loss: 1.6274 - val_accuracy: 0.2416 - val_loss: 1.5954
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.2479 - loss: 1.6213 - val_accuracy: 0.2416 - val_loss: 1

[1.5915536880493164, 0.2426966279745102]

## 기존 모델과 비교

In [40]:
models = {
    'RNN 기본'        : (model_rnn,  history_rnn),
    'RNN SpatialDrop' : (model_rnn2, history_rnn2),
}

for name, (model, hist) in models.items():
    tr  = round(hist.history['accuracy'][-1]*100, 1)
    val = round(hist.history['val_accuracy'][-1]*100, 1)
    _, te = model.evaluate(X_test, y_test, verbose=0)
    print(f"{name:20s}  Train:{tr}  Val:{val}  Test:{round(te*100,1)}")

RNN 기본                Train:100.0  Val:35.4  Test:30.8
RNN SpatialDrop       Train:27.1  Val:22.5  Test:24.3


In [ ]:
# 결론: RNN 포기!! SimpleRNN은 긴 텍스트(200 단어) 에서 장기 의존성 문제로 근본적으로 취약... 드롭아웃을 조금만 줘도 학습 자체가 안 됨 ㅜㅜ 

# 과적합 DNN

In [46]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model_dnn3 = Sequential([
    layers.Embedding(input_dim=10000, output_dim=32, input_length=200),
    layers.SpatialDropout1D(0.2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')
])
model_dnn3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_dnn3 = model_dnn3.fit(X_train, y_train, epochs=30, batch_size=64, 
                               validation_split=0.2, callbacks=[es])
model_dnn3.evaluate(X_test, y_test)

Epoch 1/30


/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1775023969.797129   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_125482__.16


12/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2075 - loss: 1.6073

I0000 00:00:1775023970.936642   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_125482__.16


23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - accuracy: 0.2781 - loss: 1.5923 - val_accuracy: 0.3455 - val_loss: 1.5551
Epoch 2/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7296 - loss: 1.3615 - val_accuracy: 0.4382 - val_loss: 1.3842
Epoch 3/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8827 - loss: 0.8233 - val_accuracy: 0.8146 - val_loss: 0.8654
Epoch 4/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9838 - loss: 0.2518 - val_accuracy: 0.8820 - val_loss: 0.4867
Epoch 5/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9972 - loss: 0.0852 - val_accuracy: 0.8876 - val_loss: 0.3780
Epoch 6/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9986 - loss: 0.0397 - val_accuracy: 0.9101 - val_loss: 0.3099
Epoch 7/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9993 - loss: 0.0246 - val_accuracy: 0.9045 - val_loss: 0.2936
Epoch 8/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0187 - val_accuracy: 0.9185 - val_loss: 0.2719
Ep

[0.2580980360507965, 0.9056179523468018]

In [47]:
names     = ['DNN 기본', 'DNN 수정']
models    = [model_dnn,   model_dnn2]
histories = [history_dnn, history_dnn2]

results = []
for name, model, hist in zip(names, models, histories):
    tr  = round(hist.history['accuracy'][-1]*100, 1)
    val = round(hist.history['val_accuracy'][-1]*100, 1)
    _, te = model.evaluate(X_test, y_test, verbose=0)
    results.append([name, tr, val, round(te*100, 1)])

df = pd.DataFrame(results, columns=['모델', 'Train Acc(%)', 'Val Acc(%)', 'Test Acc(%)'])
df

,모델,Train Acc(%),Val Acc(%),Test Acc(%)
0,DNN 기본,100.0,92.7,87.2
1,DNN 수정,99.9,89.9,84.9


# LSTM 수정 코드

In [52]:
model_lstm3 = Sequential([
    layers.Embedding(input_dim=10000, output_dim=32, input_length=200),
    layers.SpatialDropout1D(0.1),
    layers.LSTM(32, dropout=0.1, recurrent_dropout=0.1),
    layers.Dense(5, activation='softmax')
])
model_lstm3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_lstm3 = model_lstm3.fit(X_train, y_train, epochs=10, batch_size=64, 
                                 validation_split=0.2)  # callbacks 제거
model_lstm3.evaluate(X_test, y_test)

Epoch 1/10


/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


23/23 ━━━━━━━━━━━━━━━━━━━━ 18s 725ms/step - accuracy: 0.2633 - loss: 1.6006 - val_accuracy: 0.2809 - val_loss: 1.5862
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 18s 778ms/step - accuracy: 0.3364 - loss: 1.5043 - val_accuracy: 0.4466 - val_loss: 1.2407
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 717ms/step - accuracy: 0.4368 - loss: 1.2408 - val_accuracy: 0.4747 - val_loss: 1.1831
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 22s 686ms/step - accuracy: 0.4838 - loss: 1.2166 - val_accuracy: 0.4551 - val_loss: 1.1749
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 17s 753ms/step - accuracy: 0.4480 - loss: 1.1843 - val_accuracy: 0.4551 - val_loss: 1.1699
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 676ms/step - accuracy: 0.4831 - loss: 1.1551 - val_accuracy: 0.4663 - val_loss: 1.1727
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 18s 792ms/step - accuracy: 0.5864 - loss: 1.1079 - val_accuracy: 0.4803 - val_loss: 1.1181
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 680ms/step - accuracy: 0.6426 - loss: 1.0452 - val_accuracy: 0.525

[0.9344967007637024, 0.6000000238418579]

In [54]:
names     = ['LSTM 기본', 'LSTM 수정']
models    = [model_lstm,   model_lstm3]
histories = [history_lstm, history_lstm3]

results = []
for name, model, hist in zip(names, models, histories):
    tr  = round(hist.history['accuracy'][-1]*100, 1)
    val = round(hist.history['val_accuracy'][-1]*100, 1)
    _, te = model.evaluate(X_test, y_test, verbose=0)
    results.append([name, tr, val, round(te*100, 1)])

df = pd.DataFrame(results, columns=['모델', 'Train Acc(%)', 'Val Acc(%)', 'Test Acc(%)'])
df

,모델,Train Acc(%),Val Acc(%),Test Acc(%)
0,LSTM 기본,77.3,65.4,60.7
1,LSTM 수정,74.2,62.9,60.0


In [56]:
# 새 뉴스 텍스트 예측
new_text = ["The prime minister announced new economic policies today in parliament"]

# 전처리 (학습때와 동일하게)
seq = tokenizer.texts_to_sequences(new_text)
pad = pad_sequences(seq, maxlen=200, padding='post', truncating='post')

# 예측
pred = model_dnn2.predict(pad)
pred_label = le.inverse_transform([pred.argmax()])

print(f"입력 텍스트: {new_text[0]}")
print(f"예측 카테고리: {pred_label[0]}")
print(f"\n각 카테고리 확률:")
for cat, prob in zip(le.classes_, pred[0]):
    print(f"  {cat:15s}: {prob*100:.1f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
입력 텍스트: The prime minister announced new economic policies today in parliament
예측 카테고리: sport

각 카테고리 확률:
  business       : 2.5%
  entertainment  : 3.3%
  politics       : 2.5%
  sport          : 91.6%
  tech           : 0.1%


In [58]:
df_bbc = pd.read_csv('/home/ubuntu/projects/data/bbc-text.csv')
print(df_bbc['category'].value_counts())

category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [59]:
texts = [
    "Manchester United won the match against Liverpool",   # sport
    "Apple released a new iPhone with AI features",        # tech
    "The prime minister announced new economic policies",  # politics
    "The stock market crashed due to inflation concerns",  # business
    "The new movie won three Oscar awards last night",     # entertainment
]

seqs = tokenizer.texts_to_sequences(texts)
pads = pad_sequences(seqs, maxlen=200, padding='post', truncating='post')
preds = model_dnn2.predict(pads)

for text, pred in zip(texts, preds):
    label = le.inverse_transform([pred.argmax()])[0]
    print(f"예측: {label:15s} | {text[:50]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 602ms/step
예측: sport           | Manchester United won the match against Liverpool
예측: sport           | Apple released a new iPhone with AI features
예측: sport           | The prime minister announced new economic policies
예측: sport           | The stock market crashed due to inflation concerns
예측: sport           | The new movie won three Oscar awards last night


In [60]:
pred = model_dnn3.predict(pad)  # model_dnn2 → model_dnn3

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step


In [61]:
texts = [
    "Manchester United won the match against Liverpool",
    "Apple released a new iPhone with AI features",
    "The prime minister announced new economic policies",
    "The stock market crashed due to inflation concerns",
    "The new movie won three Oscar awards last night",
]

seqs = tokenizer.texts_to_sequences(texts)
pads = pad_sequences(seqs, maxlen=200, padding='post', truncating='post')
preds = model_dnn3.predict(pads)  # ← 여기 변경

for text, pred in zip(texts, preds):
    label = le.inverse_transform([pred.argmax()])[0]
    print(f"예측: {label:15s} | {text[:50]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
예측: sport           | Manchester United won the match against Liverpool
예측: sport           | Apple released a new iPhone with AI features
예측: sport           | The prime minister announced new economic policies
예측: sport           | The stock market crashed due to inflation concerns
예측: sport           | The new movie won three Oscar awards last night


In [62]:
texts = [
    "Arsenal beat Chelsea in the premier league match on saturday",           # sport
    "Microsoft launched a new software update for windows operating system",  # tech
    "Chancellor announced budget cuts to reduce government spending deficit",  # politics
    "Goldman sachs reported record profits in quarterly earnings report",      # business
    "Hollywood actress won academy award for best performance in drama film",  # entertainment
]

seqs = tokenizer.texts_to_sequences(texts)
pads = pad_sequences(seqs, maxlen=200, padding='post', truncating='post')
preds = model_dnn3.predict(pads)

for text, pred in zip(texts, preds):
    label = le.inverse_transform([pred.argmax()])[0]
    prob  = pred.max()*100
    print(f"예측: {label:15s} ({prob:.1f}%) | {text[:55]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
예측: sport           (99.7%) | Arsenal beat Chelsea in the premier league match on sat
예측: sport           (99.7%) | Microsoft launched a new software update for windows op
예측: sport           (97.9%) | Chancellor announced budget cuts to reduce government s
예측: sport           (99.4%) | Goldman sachs reported record profits in quarterly earn
예측: sport           (99.2%) | Hollywood actress won academy award for best performanc


In [63]:
model_dnn3.evaluate(X_test, y_test)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9056 - loss: 0.2581


[0.2580980360507965, 0.9056179523468018]

In [64]:
# 각 모델 Test Acc 확인
for name, m in [('dnn 기본', model_dnn), ('dnn2', model_dnn2), ('dnn3', model_dnn3)]:
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{name}: {acc:.4f}")

dnn 기본: 0.8719
dnn2: 0.8494
dnn3: 0.9056


In [66]:
preds = model_dnn.predict(pads)

for text, pred in zip(texts, preds):
    label = le.inverse_transform([pred.argmax()])[0]
    prob  = pred.max()*100
    print(f"예측: {label:15s} ({prob:.1f}%) | {text[:55]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
예측: sport           (99.5%) | Arsenal beat Chelsea in the premier league match on sat
예측: sport           (97.8%) | Microsoft launched a new software update for windows op
예측: sport           (97.4%) | Chancellor announced budget cuts to reduce government s
예측: sport           (98.7%) | Goldman sachs reported record profits in quarterly earn
예측: sport           (98.2%) | Hollywood actress won academy award for best performanc


In [67]:
# OOV 토큰이 얼마나 되는지 확인
test_text = "Microsoft launched a new software update for windows operating system"
seq = tokenizer.texts_to_sequences([test_text])
print(seq)  # 1이 OOV 토큰
print(f"OOV 비율: {seq[0].count(1)/len(seq[0])*100:.0f}%")

[[388, 745, 6, 49, 257, 3805, 9, 922, 1146, 234]]
OOV 비율: 0%


In [68]:
# 실제 X_test에서 샘플 뽑아서 예측
sample_idx = [0, 1, 2, 3, 4]
preds = model_dnn.predict(X_test[sample_idx])

for idx, pred in zip(sample_idx, preds):
    true_label = le.inverse_transform([y_test[idx]])[0]
    pred_label = le.inverse_transform([pred.argmax()])[0]
    print(f"실제: {true_label:15s} | 예측: {pred_label:15s} | {df_bbc['text'].iloc[idx][:40]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
실제: politics        | 예측: politics        | tv future in the hands of viewers with h
실제: business        | 예측: business        | worldcom boss  left books alone  former 
실제: entertainment   | 예측: entertainment   | tigers wary of farrell  gamble  leiceste
실제: tech            | 예측: tech            | yeading face newcastle in fa cup premier
실제: sport           | 예측: sport           | ocean s twelve raids box office ocean s 


In [69]:
print(le.classes_)
print(le.transform(le.classes_))

['business' 'entertainment' 'politics' 'sport' 'tech']
[0 1 2 3 4]


In [70]:
# 예측값 raw 확인
preds = model_dnn.predict(pads)
print(preds)  # 각 카테고리별 확률 원본

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[2.6887821e-03 3.9753201e-04 1.9467345e-03 9.9496680e-01 8.9143370e-08]
 [1.1829139e-02 9.7591331e-04 9.4960155e-03 9.7769827e-01 7.6194266e-07]
 [1.1283184e-02 9.3560683e-04 1.3673116e-02 9.7410786e-01 2.5738402e-07]
 [8.8246670e-03 4.9026392e-04 3.5243854e-03 9.8716050e-01 1.2650470e-07]
 [8.8298647e-03 3.7312692e-03 5.2395146e-03 9.8219925e-01 1.5759129e-07]]


In [ ]:
# 아까 실제 X_test로는 잘 예측했는데, 새 문장에서는 sport만 나온다 = 모델이 BBC 훈련 데이터의 특정 패턴을 외운 것!

In [ ]:
# 해결 - 모델을 처음부터 다시 학습

In [72]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model_final = Sequential([
    layers.Embedding(input_dim=10000, output_dim=64, input_length=200),  # 32 → 64
    layers.SpatialDropout1D(0.3),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])
model_final.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_final = model_final.fit(X_train, y_train, epochs=30, batch_size=32,   # batch 줄임
                                 validation_split=0.2, callbacks=[es])
model_final.evaluate(X_test, y_test)

Epoch 1/30


/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1775025373.528008   68924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_151596__.16


40/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2645 - loss: 1.5884

I0000 00:00:1775025376.185886   68912 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_151596__.16


45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 81ms/step - accuracy: 0.3448 - loss: 1.5493 - val_accuracy: 0.5506 - val_loss: 1.4419
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9010 - loss: 0.7693 - val_accuracy: 0.8399 - val_loss: 0.6251
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9944 - loss: 0.0928 - val_accuracy: 0.9073 - val_loss: 0.3571
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0237 - val_accuracy: 0.9185 - val_loss: 0.2973
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0132 - val_accuracy: 0.9242 - val_loss: 0.2658
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0084 - val_accuracy: 0.9242 - val_loss: 0.2466
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0063 - val_accuracy: 0.9213 - val_loss: 0.2516
Epoch 8/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0056 - val_accuracy: 0.9354 - val_loss: 0.2139
Ep

[0.2476952075958252, 0.9123595356941223]

In [73]:
texts = [
    "arsenal chelsea premier league goal scored saturday",
    "microsoft windows software update operating system",
    "government chancellor budget spending tax parliament",
    "bank profit shares market economy quarterly results",
    "film actor oscar award cinema box office released",
]

seqs = tokenizer.texts_to_sequences(texts)
pads = pad_sequences(seqs, maxlen=200, padding='post', truncating='post')
preds = model_final.predict(pads)

for text, pred in zip(texts, preds):
    label = le.inverse_transform([pred.argmax()])[0]
    prob  = pred.max()*100
    print(f"예측: {label:15s} ({prob:.1f}%) | {text}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 763ms/step
예측: sport           (99.9%) | arsenal chelsea premier league goal scored saturday
예측: sport           (99.8%) | microsoft windows software update operating system
예측: sport           (99.2%) | government chancellor budget spending tax parliament
예측: sport           (98.9%) | bank profit shares market economy quarterly results
예측: sport           (99.6%) | film actor oscar award cinema box office released


In [74]:
# y_train에서 sport 비율 확인
import numpy as np
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(le.inverse_transform(unique), counts):
    print(f"{u:15s}: {c}")

business       : 409
entertainment  : 305
politics       : 334
sport          : 413
tech           : 319


In [75]:
# 각 카테고리에서 1개씩 실제 텍스트 뽑기
for cat in ['sport', 'tech', 'business', 'politics', 'entertainment']:
    sample = df_bbc[df_bbc['category'] == cat]['text'].iloc[0][:200]
    
    seq = tokenizer.texts_to_sequences([sample])
    pad = pad_sequences(seq, maxlen=200, padding='post', truncating='post')
    pred = model_final.predict(pad, verbose=0)
    label = le.inverse_transform([pred.argmax()])[0]
    prob = pred.max()*100
    print(f"실제: {cat:15s} | 예측: {label:15s} ({prob:.1f}%)")

실제: sport           | 예측: sport           (100.0%)
실제: tech            | 예측: sport           (99.7%)
실제: business        | 예측: sport           (99.3%)
실제: politics        | 예측: sport           (61.2%)
실제: entertainment   | 예측: sport           (99.8%)


In [76]:
for cat in ['sport', 'tech', 'business', 'politics', 'entertainment']:
    sample = df_bbc[df_bbc['category'] == cat]['text'].iloc[0][:200]
    seq = tokenizer.texts_to_sequences([sample])
    pad = pad_sequences(seq, maxlen=200, padding='post', truncating='post')
    pred = model_dnn.predict(pad, verbose=0)
    label = le.inverse_transform([pred.argmax()])[0]
    prob = pred.max()*100
    print(f"실제: {cat:15s} | 예측: {label:15s} ({prob:.1f}%)")


실제: sport           | 예측: sport           (99.7%)
실제: tech            | 예측: sport           (96.0%)
실제: business        | 예측: sport           (97.5%)
실제: politics        | 예측: sport           (84.0%)
실제: entertainment   | 예측: sport           (96.2%)


In [77]:
sample_idx = [0, 1, 2, 3, 4]
preds = model_dnn.predict(X_test[sample_idx])

for idx, pred in zip(sample_idx, preds):
    true_label = le.inverse_transform([y_test[idx]])[0]
    pred_label = le.inverse_transform([pred.argmax()])[0]
    print(f"실제: {true_label:15s} | 예측: {pred_label:15s}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
실제: politics        | 예측: politics       
실제: business        | 예측: business       
실제: entertainment   | 예측: entertainment  
실제: tech            | 예측: tech           
실제: sport           | 예측: sport          


In [78]:
sample_idx = list(range(0, 50, 10))  # 10개 간격으로 샘플
preds = model_dnn.predict(X_test[sample_idx])

for idx, pred in zip(sample_idx, preds):
    true_label = le.inverse_transform([y_test[idx]])[0]
    pred_label = le.inverse_transform([pred.argmax()])[0]
    correct = "✅" if true_label == pred_label else "❌"
    print(f"{correct} 실제: {true_label:15s} | 예측: {pred_label:15s}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
✅ 실제: politics        | 예측: politics       
✅ 실제: entertainment   | 예측: entertainment  
❌ 실제: entertainment   | 예측: business       
✅ 실제: politics        | 예측: politics       
✅ 실제: business        | 예측: business       


BBC 뉴스 데이터 내에서는 90% 정확도로 분류 가능하지만, 데이터 수가 적어 새로운 문장에 대한 일반화는 어렵다...ㅜㅜ